In [1]:
import os
import requests
import json
from dotenv import load_dotenv
# from langchain_openai import ChatOpenAI
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_experimental.sql import SQLDatabaseSequentialChain
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_core.prompts import MessagesPlaceholder, ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
from typing_extensions import Annotated, TypedDict
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
# from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor
from langchain_openai import ChatOpenAI
import psycopg2
import time 
import yaml
import json 
from langchain_community.agent_toolkits import JsonToolkit, create_json_agent
from langchain_community.tools.json.tool import JsonSpec
from langchain_openai import OpenAI
from langchain.tools import Tool

### Add Thingsboard details

In [2]:
THINGSBOARD_URL = "http://localhost:8080"
USERNAME = "tenant@thingsboard.org"
PASSWORD = "tenant"
DASHBOARD_ID = "http://localhost:8080/tenants"

DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "thingsboard"
DB_USER = "thingsboard"
DB_PASSWORD = "postgres"


THINGSBOARD_HOST = "http://localhost:8080"
USERNAME = "tenant@thingsboard.org"
PASSWORD = "tenant"

# Authenticate and get the JWT token
auth_url = f'{THINGSBOARD_HOST}/api/auth/login'
auth_payload = {'username': USERNAME, 'password': PASSWORD}
auth_response = requests.post(auth_url, json=auth_payload)
auth_response.raise_for_status()
jwt_token = auth_response.json()['token']

In [3]:
# Load environment variables
load_dotenv()
openai_api_key = os.getenv('OPENAI_PROJECT_API_KEY')

In [4]:
# Load the device metadata
import json 
with open("farm_model_small_v2.json", "r") as file:
    data = json.load(file)

In [61]:
# data['farm']['fields'][0]

In [7]:
def get_farm_details(a: int) -> str:
    """Return Farm field details as a JSON string

    Args:
        a: Field Index
    """
    try:
        field_data = data['farm']['fields'][a]
        return json.dumps(field_data)
    except IndexError:
        return json.dumps({"error": f"Field index {a} is out of bounds."})
    except KeyError:
        return json.dumps({"error": "The 'farm' or 'fields' key was not found in the data."})
    except Exception as e:
        return json.dumps({"error": f"An error occurred: {e}"})

In [9]:
field_index = 0
farm_details_json = get_farm_details(field_index)
print(farm_details_json)

{"F001": {"name": "North Field", "crop": "Maize", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0100"], "field_air_humidity": ["HUM-0100"], "soil_conductivity": ["COND-0100"], "moisture_content": ["MOIST-0100"], "plant_health": ["CAM-0100"]}, "actuator_list": {"pumps": ["PUMP-0100", "PUMP-0101"], "water_valves": ["WV-0100"], "fertilizer_dispensers": ["FD-0100"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0100", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air_humidity": [{"sensor_id": "HUM-0100", "gps": {"lat": 35.6802, "long": -98.1202}, "status": "transmitting", "unit": "grams/cubic meter"}], "soil_conductivity": [{"sensor_id": "COND-0100", "gps": {"lat": 35.6803, "long": -98.1203}, "status": "transmitting", "u

In [10]:
field_index = 6# Assuming there are fewer than 6 fields
farm_details_json = get_farm_details(field_index)
print(farm_details_json)

{"F007": {"name": "West Field", "crop": "Coffee", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0700"], "field_air_humidity": ["HUM-0700"], "soil_conductivity": ["COND-0700"], "moisture_content": ["MOIST-0700"], "plant_health": ["CAM-0700"]}, "actuator_list": {"pumps": ["PUMP-0700", "PUMP-0701"], "water_valves": ["WV-0700"], "fertilizer_dispensers": ["FD-0700"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0700", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air_humidity": [{"sensor_id": "HUM-0700", "gps": {"lat": 35.6802, "long": -98.1202}, "status": "transmitting", "unit": "grams/cubic meter"}], "soil_conductivity": [{"sensor_id": "COND-0700", "gps": {"lat": 35.6803, "long": -98.1203}, "status": "transmitting", "u

In [12]:
# create tool
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_farm_details",
            "description": "Return details about a specific farm field as a JSON object.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {
                        "type": "integer",
                        "description": "The index of the field to retrieve details for (0-based).",
                    },
                },
                "required": ["a"],
            },
        },
    }
]

In [13]:
import openai 
# client = openai.OpenAI(api_key=openai_api_key)
client = openai.OpenAI(api_key=openai_api_key)
llm_model = "gpt-3.5-turbo"
# client = ChatOpenAI(api_key=openai_api_key, temperature = 0.0, model=llm_model)
# client = openai.OpenAI(api_key=openai_api_key,model=llm_model)

In [14]:
def run_conversation(query):
    """Runs a conversation with the LLM that can call the get_farm_details tool."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant that identifies relevant devices (sensor_list) mentioned in the user's request and returns them as a JSON array of their IDs e.g. If no specific devices are mentioned, return an empty JSON array."
        "F001 is North Field,"
        "F002 is Northeast Field"
        "F003 is East Field"
        "F004 is Southeast Field"
        "F005 is South Field"
        "F006 is Southwest Field"
        "F007 is West Field"
        "F008 is Northwest Field"
        "F009 is Central Field"
        "Note: if the right Field is not provided return an empty json"
        },
     
        {"role": "user", "content": query}
    ]

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        tools=tools,
        tool_choice="auto", 
    )

    response_message = response.choices[0].message

    if response_message.tool_calls:
        print("LLM initiated a tool call:")
        print(response_message.tool_calls)

        tool_call = response_message.tool_calls[0]
        function_name = tool_call.function.name
        function_to_call = globals()[function_name]
        function_args = json.loads(tool_call.function.arguments)
        function_response = function_to_call(**function_args)

        print(f"Calling function '{function_name}' with arguments: {function_args}")
        print(f"Function returned: {function_response}")

        messages.append(response_message)
        messages.append(
            {
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": function_response,
            }
        )
        second_response = client.chat.completions.create(
            # model="gpt-3.5-turbo-0613",
            model="gpt-3.5-turbo",
            messages=messages,
        )
        
        return second_response.choices[0].message.content
    else:
        # If no tool call, the LLM might be directly answering based on the system prompt
        try:
            # Attempt to parse the response as JSON (assuming it followed the system prompt)
            return json.loads(response_message.content)
        except (json.JSONDecodeError, TypeError):
            # If it's not valid JSON, return the raw content
            return response_message.content

In [15]:
user_query_farm = "Tell me the details of the sensors in the south  field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_HaEwhSGvMGTntg8vfoiYFfCg', function=Function(arguments='{"a":4}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 4}
Function returned: {"F005": {"name": "South Field", "crop": "soybean", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0500"], "field_air_humidity": ["HUM-0500"], "soil_conductivity": ["COND-0500"], "moisture_content": ["MOIST-0500"], "plant_health": ["CAM-0500"]}, "actuator_list": {"pumps": ["PUMP-0500", "PUMP-0501"], "water_valves": ["WV-0500"], "fertilizer_dispensers": ["FD-0500"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0500", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air

In [35]:
user_query_farm = "Tell me the details of the sensors in the east field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_yD3QENlTlz9B9jO3WJkF4UZl', function=Function(arguments='{"a":2}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 2}
Function returned: {"F003": {"name": "East Field", "crop": "Sorghum", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0300"], "field_air_humidity": ["HUM-0300"], "soil_conductivity": ["COND-0300"], "moisture_content": ["MOIST-0300"], "plant_health": ["CAM-0300"]}, "actuator_list": {"pumps": ["PUMP-0300", "PUMP-0301"], "water_valves": ["WV-0300"], "fertilizer_dispensers": ["FD-0300"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0300", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air_

In [36]:
user_query_farm = "Tell me the details of the sensors in the give South field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_gIEAxXoobBIphLE7CFzcGOsW', function=Function(arguments='{"a":4}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 4}
Function returned: {"F005": {"name": "South Field", "crop": "soybean", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0500"], "field_air_humidity": ["HUM-0500"], "soil_conductivity": ["COND-0500"], "moisture_content": ["MOIST-0500"], "plant_health": ["CAM-0500"]}, "actuator_list": {"pumps": ["PUMP-0500", "PUMP-0501"], "water_valves": ["WV-0500"], "fertilizer_dispensers": ["FD-0500"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0500", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air

In [46]:
user_query_farm = "Tell me the details of the temperature sensor in the give South field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_sNeCQsAilt5bL8s7q1S7moLH', function=Function(arguments='{"a":4}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 4}
Function returned: {"F005": {"name": "South Field", "crop": "soybean", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0500"], "field_air_humidity": ["HUM-0500"], "soil_conductivity": ["COND-0500"], "moisture_content": ["MOIST-0500"], "plant_health": ["CAM-0500"]}, "actuator_list": {"pumps": ["PUMP-0500", "PUMP-0501"], "water_valves": ["WV-0500"], "fertilizer_dispensers": ["FD-0500"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0500", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air

In [38]:
user_query_farm = "Tell me the details of the temperature sensor in the give South-West field."
response_farm = run_conversation(user_query_farm)
print(f"\nLLM Response about farm: {response_farm}")


LLM initiated a tool call:
[ChatCompletionMessageToolCall(id='call_fgAEyGL2Qd2zMUb5e03702Rw', function=Function(arguments='{"a":5}', name='get_farm_details'), type='function')]
Calling function 'get_farm_details' with arguments: {'a': 5}
Function returned: {"F006": {"name": "Southwest Field", "crop": "Soybean", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0600"], "field_air_humidity": ["HUM-0600"], "soil_conductivity": ["COND-0600"], "moisture_content": ["MOIST-0600"], "plant_health": ["CAM-0600"]}, "actuator_list": {"pumps": ["PUMP-0600", "PUMP-0601"], "water_valves": ["WV-0600"], "fertilizer_dispensers": ["FD-0600"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0600", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field

In [47]:
sensor_ids = json.loads(response_farm)["sensor_list"]
sensor_ids

['TEMP-0500']

In [48]:
# 2. Get device ID by name
def get_device_id_by_name(device_name, token):
    headers = {
        "Content-Type": "application/json",
        "X-Authorization": f"Bearer {token}"
    }
    url = f"{THINGSBOARD_URL}/api/tenant/devices?deviceName={device_name}"
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    device = response.json()
    return device['id']['id'] if device else None

In [49]:
def get_device_keys(jwt_token, device_id):
    url = f"{THINGSBOARD_URL}/api/plugins/telemetry/DEVICE/{device_id}/keys/timeseries"
    headers = {
        "X-Authorization": f"Bearer {jwt_token}"
    }

    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()  # Returns a list of key names
    else:
        return {
            "error": f"Failed to fetch keys: {response.status_code}",
            "details": response.text
        }

In [50]:
# change the device name 
device_name = sensor_ids[0] #"HUM-0100"
sensor_id = get_device_id_by_name(device_name, jwt_token)
device_id = sensor_id
sensor_id

'e82fc1b0-1221-11f0-a236-5f0808fc8cdf'

In [51]:
device_key = get_device_keys(jwt_token, sensor_id)
device_key = device_key[-1] # ['deviceId', 'unit', 'relative_humidity']
device_key 

'temp'

# Get data for the last 24 hours

In [52]:
# Last 24 hours timestamps
end_ts = int(time.time() * 1000)  
start_ts = end_ts - (340 * 60 * 60 * 1000)  # 24 hours ago

# keys = "temperature"  
keys = device_key


def get_historical_data(jwt_token, device_id, start_ts, end_ts, keys):
    url = f"{THINGSBOARD_URL}/api/plugins/telemetry/DEVICE/{device_id}/values/timeseries"
    params = {"keys": keys, "startTs": start_ts, "endTs": end_ts, "limit": 100}
    headers = {"X-Authorization": f"Bearer {jwt_token}"}
    
    response = requests.get(url, headers=headers, params=params)
    return response.json() if response.status_code == 200 else {"error": "Failed to fetch telemetry data"}

response = get_historical_data(jwt_token, sensor_id, start_ts, end_ts, keys)
response

{'temp': [{'ts': 1743869011403, 'value': '27.320302266151312'},
  {'ts': 1743868709415, 'value': '31.306305009158017'},
  {'ts': 1743868407223, 'value': '33.48834158762985'},
  {'ts': 1743868105251, 'value': '34.10189227675727'},
  {'ts': 1743867803070, 'value': '28.534077498339663'},
  {'ts': 1743867501220, 'value': '27.817652246517067'},
  {'ts': 1743867199515, 'value': '32.49147749209336'},
  {'ts': 1743866897596, 'value': '28.854327077358164'},
  {'ts': 1743866595362, 'value': '28.009212122493373'},
  {'ts': 1743866293508, 'value': '31.45880622072238'},
  {'ts': 1743865991358, 'value': '33.49596248476476'},
  {'ts': 1743864788511, 'value': '25.02549472745727'}]}

In [53]:
telemetry_data = response

In [ ]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import CSVLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch
from langchain.evaluation.qa import QAGenerateChain

llm_model = "gpt-3.5-turbo"
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(api_key=openai_api_key,model=llm_model))

/var/folders/bw/zwn916250j389j86x0z9f6tr0000gn/T/ipykernel_40224/16338956.py:10: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(api_key=openai_api_key,model=llm_model))


In [62]:
import langchain 
langchain.debug = True
docs = [{"doc": json.dumps(data['farm']['fields'][i])} for i in range(len(data['farm']['fields']))]
docs

[{'doc': '{"F001": {"name": "North Field", "crop": "Maize", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0100"], "field_air_humidity": ["HUM-0100"], "soil_conductivity": ["COND-0100"], "moisture_content": ["MOIST-0100"], "plant_health": ["CAM-0100"]}, "actuator_list": {"pumps": ["PUMP-0100", "PUMP-0101"], "water_valves": ["WV-0100"], "fertilizer_dispensers": ["FD-0100"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0100", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air_humidity": [{"sensor_id": "HUM-0100", "gps": {"lat": 35.6802, "long": -98.1202}, "status": "transmitting", "unit": "grams/cubic meter"}], "soil_conductivity": [{"sensor_id": "COND-0100", "gps": {"lat": 35.6803, "long": -98.1203}, "status": "transmi

In [64]:
new_examples = example_gen_chain.apply_and_parse(docs)

/Users/george/Downloads/eai_methods/env/lib/python3.10/site-packages/langchain/chains/llm.py:369: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


[chain/start] [chain:QAGenerateChain] Entering Chain run with input:
{
  "input_list": [
    {
      "doc": "{\"F001\": {\"name\": \"North Field\", \"crop\": \"Maize\", \"area\": \"62.5 acres\", \"boundary_gps\": {\"north\": {\"lat\": 35.6789, \"long\": -98.1234}, \"east\": {\"lat\": 35.6789, \"long\": -98.1234}, \"south\": {\"lat\": 35.6789, \"long\": -98.1234}, \"west\": {\"lat\": 35.6789, \"long\": -98.1234}}, \"sensor_list\": {\"soil_temperature\": [\"TEMP-0100\"], \"field_air_humidity\": [\"HUM-0100\"], \"soil_conductivity\": [\"COND-0100\"], \"moisture_content\": [\"MOIST-0100\"], \"plant_health\": [\"CAM-0100\"]}, \"actuator_list\": {\"pumps\": [\"PUMP-0100\", \"PUMP-0101\"], \"water_valves\": [\"WV-0100\"], \"fertilizer_dispensers\": [\"FD-0100\"]}, \"sensors\": {\"soil_temperature\": [{\"sensor_id\": \"TEMP-0100\", \"gps\": {\"lat\": 35.6801, \"long\": -98.1201}, \"status\": \"transmitting\", \"unit\": \"celsius\"}], \"field_air_humidity\": [{\"sensor_id\": \"HUM-0100\", \"gps

In [72]:
new_ex0 = [{'qa_pairs': {'query': 'What is the crop being grown in the "North Field" and what is the area of the field?',
   'answer': 'The crop being grown in the "North Field" is Maize, and the area of the field is 62.5 acres.'}},
 {'qa_pairs': {'query': 'What is the crop being grown in the Northeast Field and what are the sensor and actuator types associated with this field?',
   'answer': 'The crop being grown in the Northeast Field is Maize. The sensors associated with this field are soil temperature sensor (TEMP-0200), field air humidity sensor (HUM-0200), soil conductivity sensor (COND-0200), moisture content sensor (MOIST-0200), and plant health camera (CAM-0200). The actuators associated with this field are pumps (PUMP-0200, PUMP-0201), water valve (WV-0200), and fertilizer dispenser (FD-0200).'}},
 {'qa_pairs': {'query': 'What is the name of the field mentioned in the document and what is the crop being grown in that field?',
   'answer': 'The name of the field is "East Field" and the crop being grown in that field is "Sorghum".'}},
 {'qa_pairs': {'query': 'What is the name of the field mentioned in the document and what crop is being grown on it?',
   'answer': 'The name of the field is "Southeast Field" and the crop being grown on it is Maize.'}},
 {'qa_pairs': {'query': 'What is the name of the field mentioned in the document and what crop is being grown there?',
   'answer': 'The name of the field is "South Field" and the crop being grown there is soybean.'}},
 {'qa_pairs': {'query': 'What is the name of the field mentioned in the document and what crop is being grown there?',
   'answer': 'The name of the field is "Southwest Field" and the crop being grown there is Soybean.'}},
 {'qa_pairs': {'query': 'What is the name of the field mentioned in the document and what is the crop being grown in that field?',
   'answer': 'The name of the field is "West Field" and the crop being grown in that field is "Coffee".'}},
 {'qa_pairs': {'query': 'What is the crop being grown in Northwest Field and what are the units of measurement for the soil temperature sensor in that area?',
   'answer': 'The crop being grown in Northwest Field is Potato. The unit of measurement for the soil temperature sensor in that area is Celsius.'}},
 {'qa_pairs': {'query': 'What is the name of the field, the crop being grown, and the total area of Central Field based on the provided document?',
   'answer': 'The name of the field is Central Field, the crop being grown is Potato, and the total area is 62.5 acres.'}}]

In [73]:
new_ex1 = [{'qa_pairs': {'query': 'What is the crop being grown in the North Field and how many acres does it cover?',
   'answer': 'The crop being grown in the North Field is Maize and it covers 62.5 acres.'}},
 {'qa_pairs': {'query': 'What is the name of the field mentioned in the document and what crop is being grown in that field?',
   'answer': 'The name of the field is "Northeast Field" and the crop being grown is Maize.'}},
 {'qa_pairs': {'query': 'What is the name of the field mentioned in the document and what crop is being grown in that field?',
   'answer': 'The name of the field is "East Field" and the crop being grown in that field is "Sorghum".'}},
 {'qa_pairs': {'query': 'What is the name of the field in the document, and what crop is being grown there?',
   'answer': 'The name of the field is "Southeast Field" and the crop being grown is Maize.'}},
 {'qa_pairs': {'query': 'What is the name of the field in the document and what type of crop is being grown in it?',
   'answer': 'The name of the field is "South Field" and the crop being grown in it is soybean.'}},
 {'qa_pairs': {'query': 'What is the name of the field mentioned in the document and what crop is being grown there?',
   'answer': 'The name of the field is Southwest Field and the crop being grown there is Soybean.'}},
 {'qa_pairs': {'query': 'What is the crop being grown in West Field and what are the sensor and actuator devices being used in this field?',
   'answer': 'The crop being grown in West Field is Coffee. The sensor devices being used are soil temperature sensor (TEMP-0700), field air humidity sensor (HUM-0700), soil conductivity sensor (COND-0700), moisture content sensor (MOIST-0700), and plant health camera (CAM-0700). The actuator devices being used are pumps (PUMP-0700, PUMP-0701), water valve (WV-0700), and fertilizer dispenser (FD-0700).'}},
 {'qa_pairs': {'query': 'What is the name of the field mentioned in the document and what crop is being grown there?',
   'answer': 'The name of the field is "Northwest Field" and the crop being grown there is "Potato".'}},
 {'qa_pairs': {'query': 'What is the crop being grown in Central Field and how many acres is the field?',
   'answer': 'The crop being grown in Central Field is Potato, and the field covers 62.5 acres.'}}]

In [74]:
new_ex0 += new_ex1

In [75]:
len(new_ex0)

18

In [68]:
# Turn off the debug mode
langchain.debug = False

In [ ]:
from langchain.evaluation.qa import QAEvalChain

# Create the evaluation chain
eval_chain = QAEvalChain.from_llm(model)

In [77]:
from langchain.document_loaders import JSONLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch
from langchain.embeddings import OpenAIEmbeddings

# Define the path to your JSON file
json_file_path = 'farm_model_small_v2.json' 

# Define a metadata extraction function
def metadata_func(record: dict, metadata: dict) -> dict:
    field_id = record.get("field_id", "unknown")
    metadata["field_id"] = field_id
    metadata["field_name"] = record.get("name", "unknown")
    return metadata

# Initialize the JSONLoader
loader = JSONLoader(
    file_path=json_file_path,
    jq_schema='.farm.fields[] | to_entries[] | .value',
    content_key='name',
    is_content_key_jq_parsable=False,
    metadata_func=metadata_func,
    text_content=True
)

# Load the documents
documents = loader.load()

print(documents)


[Document(metadata={'source': '/Users/george/Documents/final_push/function-calling-for-sensors-at-the-edge/farm_model_small_v2.json', 'seq_num': 1, 'field_id': 'unknown', 'field_name': 'North Field'}, page_content='North Field'), Document(metadata={'source': '/Users/george/Documents/final_push/function-calling-for-sensors-at-the-edge/farm_model_small_v2.json', 'seq_num': 2, 'field_id': 'unknown', 'field_name': 'Northeast Field'}, page_content='Northeast Field'), Document(metadata={'source': '/Users/george/Documents/final_push/function-calling-for-sensors-at-the-edge/farm_model_small_v2.json', 'seq_num': 3, 'field_id': 'unknown', 'field_name': 'East Field'}, page_content='East Field'), Document(metadata={'source': '/Users/george/Documents/final_push/function-calling-for-sensors-at-the-edge/farm_model_small_v2.json', 'seq_num': 4, 'field_id': 'unknown', 'field_name': 'Southeast Field'}, page_content='Southeast Field'), Document(metadata={'source': '/Users/george/Documents/final_push/func

In [80]:
# Create the vector store index
index_creator = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=OpenAIEmbeddings(api_key=openai_api_key)
)
vectorstore = index_creator.from_documents(documents)


/var/folders/bw/zwn916250j389j86x0z9f6tr0000gn/T/ipykernel_40224/1386404161.py:4: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding=OpenAIEmbeddings(api_key=openai_api_key)
/Users/george/Downloads/eai_methods/env/lib/python3.10/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [85]:
# index_creator = VectorstoreIndexCreator(vectorstore_cls=DocArrayInMemorySearch)
index = index_creator.from_documents(documents)

# Initialize the language model
llm = ChatOpenAI(api_key=openai_api_key, model="gpt-3.5-turbo")

# This is the correct way to get the retriever
retrieval_qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=index.vectorstore.as_retriever()
)


In [86]:
# Extract the QA pairs
qa_pairs = [output['qa_pairs'] for output in new_ex0]

In [87]:
# Generate model predictions
model_predictions = []
for pair in qa_pairs:
    question = pair['query']
    result = retrieval_qa.run(question)
    model_predictions.append({'result': result})


/var/folders/bw/zwn916250j389j86x0z9f6tr0000gn/T/ipykernel_40224/1311013834.py:5: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = retrieval_qa.run(question)


In [88]:
from langchain.evaluation.qa import QAEvalChain

# Create the evaluation chain
eval_chain = QAEvalChain.from_llm(llm)

# Evaluate the predictions
graded_outputs = eval_chain.evaluate(qa_pairs, model_predictions)

# Display the evaluation results
for i, graded in enumerate(graded_outputs):
    print(f"Example {i+1}:")
    print(f"Question: {qa_pairs[i]['query']}")
    print(f"Reference Answer: {qa_pairs[i]['answer']}")
    print(f"Model Prediction: {model_predictions[i]['result']}")
    print(f"Evaluation Result: {graded}")
    print("-" * 50)


Example 1:
Question: What is the crop being grown in the "North Field" and what is the area of the field?
Reference Answer: The crop being grown in the "North Field" is Maize, and the area of the field is 62.5 acres.
Model Prediction: I don't have specific information about the crop being grown in the "North Field" or the exact area of the field.
Evaluation Result: {'results': 'INCORRECT'}
--------------------------------------------------
Example 2:
Question: What is the crop being grown in the Northeast Field and what are the sensor and actuator types associated with this field?
Reference Answer: The crop being grown in the Northeast Field is Maize. The sensors associated with this field are soil temperature sensor (TEMP-0200), field air humidity sensor (HUM-0200), soil conductivity sensor (COND-0200), moisture content sensor (MOIST-0200), and plant health camera (CAM-0200). The actuators associated with this field are pumps (PUMP-0200, PUMP-0201), water valve (WV-0200), and fertiliz

In [89]:
from langchain.chat_models import ChatOpenAI

# Initialize the language model
llm = ChatOpenAI(api_key=openai_api_key, model="gpt-3.5-turbo")

# Define the sensor data
# sensor_data = [
#     {'sensor_id': 'TEMP-0100', 'sensor_type': 'Soil Temperature'},
#     {'sensor_id': 'HUM-0100', 'sensor_type': 'Field Air Humidity'},
#     {'sensor_id': 'COND-0100', 'sensor_type': 'Soil Conductivity'},
#     {'sensor_id': 'MOIST-0100', 'sensor_type': 'Moisture Content'},
#     {'sensor_id': 'CAM-0100', 'sensor_type': 'Plant Health Camera'}
# ]
sensor_data = data['farm']['fields']

# Define the prompt to ask the LLM to generate questions
prompt = f"""
I have a list of sensors with their IDs and types:

{sensor_data}

Generate a set of questions based on the sensor types and IDs. The questions should focus on identifying the sensor type, sensor ID, or a specific characteristic of each sensor.
For example:
- What is the sensor ID for the Soil Temperature sensor?
- Which sensor type corresponds to the ID TEMP-0100?
- What is the sensor ID for the Moisture Content sensor?

Please provide a list of questions based on this information.
"""

# Ask the LLM to generate the questions
generated_questions = llm.predict(prompt)

# Print out the generated questions
print(generated_questions)


/var/folders/bw/zwn916250j389j86x0z9f6tr0000gn/T/ipykernel_40224/732423352.py:32: LangChainDeprecationWarning: The method `BaseChatModel.predict` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  generated_questions = llm.predict(prompt)


1. What is the sensor ID for the Soil Temperature sensor in the North Field?
2. Which sensor type corresponds to the ID TEMP-0200 in the Northeast Field?
3. What is the sensor ID for the Moisture Content sensor in the East Field?
4. Which sensor type corresponds to the ID HUM-0400 in the Southeast Field?
5. What is the sensor ID for the Plant Health Camera sensor in the South Field?
6. Which sensor type corresponds to the ID COND-0600 in the Southwest Field?
7. What is the sensor ID for the Field Air Humidity sensor in the West Field?
8. Which sensor type corresponds to the ID MOIST-0800 in the Northwest Field?
9. What is the sensor ID for the Soil Conductivity sensor in the Central Field?
10. Which sensor type corresponds to the ID CAM-0900 in the Central Field?


In [90]:
generated_questions

'1. What is the sensor ID for the Soil Temperature sensor in the North Field?\n2. Which sensor type corresponds to the ID TEMP-0200 in the Northeast Field?\n3. What is the sensor ID for the Moisture Content sensor in the East Field?\n4. Which sensor type corresponds to the ID HUM-0400 in the Southeast Field?\n5. What is the sensor ID for the Plant Health Camera sensor in the South Field?\n6. Which sensor type corresponds to the ID COND-0600 in the Southwest Field?\n7. What is the sensor ID for the Field Air Humidity sensor in the West Field?\n8. Which sensor type corresponds to the ID MOIST-0800 in the Northwest Field?\n9. What is the sensor ID for the Soil Conductivity sensor in the Central Field?\n10. Which sensor type corresponds to the ID CAM-0900 in the Central Field?'

In [97]:
sensor_data = data['farm']['fields']
sensor_data

[{'F001': {'name': 'North Field',
   'crop': 'Maize',
   'area': '62.5 acres',
   'boundary_gps': {'north': {'lat': 35.6789, 'long': -98.1234},
    'east': {'lat': 35.6789, 'long': -98.1234},
    'south': {'lat': 35.6789, 'long': -98.1234},
    'west': {'lat': 35.6789, 'long': -98.1234}},
   'sensor_list': {'soil_temperature': ['TEMP-0100'],
    'field_air_humidity': ['HUM-0100'],
    'soil_conductivity': ['COND-0100'],
    'moisture_content': ['MOIST-0100'],
    'plant_health': ['CAM-0100']},
   'actuator_list': {'pumps': ['PUMP-0100', 'PUMP-0101'],
    'water_valves': ['WV-0100'],
    'fertilizer_dispensers': ['FD-0100']},
   'sensors': {'soil_temperature': [{'sensor_id': 'TEMP-0100',
      'gps': {'lat': 35.6801, 'long': -98.1201},
      'status': 'transmitting',
      'unit': 'celsius'}],
    'field_air_humidity': [{'sensor_id': 'HUM-0100',
      'gps': {'lat': 35.6802, 'long': -98.1202},
      'status': 'transmitting',
      'unit': 'grams/cubic meter'}],
    'soil_conductivity': 

In [ ]:
for field in sensor_data:
    for field_id, details in field.items():
        print(f"\n=== Field ID: {field_id} ===")
        print(f"Name: {details['name']}")
        print(f"Crop: {details['crop']}")
        print(f"Area: {details['area']}")
        print("Boundary GPS:")
        for direction, coords in details['boundary_gps'].items():
            print(f"  {direction.capitalize()}: Lat {coords['lat']}, Long {coords['long']}")

        print("\nSensors:")
        for sensor_type, sensors in details.get('sensors', {}).items():
            print(f"  {sensor_type.capitalize()}:")
            for sensor in sensors:
                sensor_id = sensor.get('sensor_id') or sensor.get('camera_id') or 'N/A'
                print(f"    ID: {sensor_id}, Status: {sensor['status']}, Unit: {sensor['unit']}, GPS: ({sensor['gps']['lat']}, {sensor['gps']['long']})")

        print("\nActuators:")
        for actuator_type, actuators in details.get('actuators', {}).items():
            print(f"  {actuator_type.capitalize()}:")
            for actuator in actuators:
                actuator_id = actuator.get('pump_id') or actuator.get('valve_id') or 'N/A'
                print(f"    ID: {actuator_id}, Type: {actuator['type']}, Status: {actuator['status']}")
                if 'linked_valves' in actuator:
                    print(f"      Linked Valves: {', '.join(actuator['linked_valves'])}")
                if 'operation_type' in actuator:
                    print(f"      Operation Type: {actuator['operation_type']}")
                if 'base_speed' in actuator:
                    print(f"      Base Speed: {actuator['base_speed']}")



=== Field ID: F001 ===
Name: North Field
Crop: Maize
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0100, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0100, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0100, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0100, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0100, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0100, Type: automated, Status: close
      Linked Valves: WV-0100
      Base Speed: 500 L/hr
    ID: PUMP-0101, Type: manual, Status: close
      Linked Valves: FD-0100
      Base Speed: 500 

In [95]:
from langchain.chat_models import ChatOpenAI

# Initialize the LLM
llm = ChatOpenAI(api_key=openai_api_key, model="gpt-3.5-turbo")

# Sensor data snippet
sensor_data = [
    {'sensor_id': 'TEMP-0100', 'sensor_type': 'Soil Temperature'},
    {'sensor_id': 'HUM-0100', 'sensor_type': 'Field Air Humidity'},
    {'sensor_id': 'COND-0100', 'sensor_type': 'Soil Conductivity'},
    {'sensor_id': 'MOIST-0100', 'sensor_type': 'Moisture Content'},
    {'sensor_id': 'CAM-0100', 'sensor_type': 'Plant Health Camera'}
]

# Format data into a string for the prompt
sensor_info = "\n".join(
    [f"- Sensor ID: {s['sensor_id']}, Type: {s['sensor_type']}" for s in sensor_data]
)

# Prompt to guide the LLM
prompt = f"""
Here is a list of sensors used in a smart farm system:

{sensor_info}

Please generate 10 different question-answer pairs that focus specifically on:
1. The sensor ID given the sensor type.
2. The sensor type given the sensor ID.
3. Other related sensor identity questions.

Format your response as a list of dictionaries like this:
{{"query": "...", "answer": "..."}}
"""

# Generate question-answer pairs
response = llm.predict(prompt)
print(response)


1. {"query": "What is the sensor ID for Soil Temperature sensor?", "answer": "Sensor ID: TEMP-0100"}
2. {"query": "What type of sensor does Sensor ID HUM-0100 represent?", "answer": "Field Air Humidity"}
3. {"query": "What type of sensor is used to measure soil conductivity?", "answer": "Sensor ID: COND-0100"}
4. {"query": "Which sensor is identified by Sensor ID MOIST-0100?", "answer": "Moisture Content sensor"}
5. {"query": "What type of sensor is represented by Sensor ID CAM-0100?", "answer": "Plant Health Camera"}
6. {"query": "What is the sensor type for Sensor ID TEMP-0100?", "answer": "Soil Temperature"}
7. {"query": "Which sensor measures air humidity in the field?", "answer": "Sensor ID: HUM-0100"}
8. {"query": "What type of data does the Soil Conductivity sensor provide?", "answer": "Soil conductivity measurements"}
9. {"query": "Which sensor is used to monitor the moisture content of the soil?", "answer": "Sensor ID: MOIST-0100"}
10. {"query": "What type of sensor is equippe

In [104]:
fields = data['farm']['fields']

In [ ]:
from langchain.chat_models import ChatOpenAI

# Initialize the language model
chat_model = ChatOpenAI(api_key=openai_api_key, model="gpt-3.5-turbo")


# Assumed: the detailed 'fields' list (F001 to F006) is already defined as per your earlier JSON-like structure

# Construct sensor info string for the LLM prompt
field_sensor_details = ""
for field in fields:
    for field_id, field_info in field.items():
        crop = field_info['crop']
        field_sensor_details += f"Field ID: {field_id}, Crop: {crop}\n"

        for sensor_category, sensors in field_info['sensors'].items():
            for sensor in sensors:
                sensor_id = sensor.get('sensor_id') or sensor.get('camera_id')
                sensor_type = sensor_category.replace('_', ' ').capitalize()
                field_sensor_details += f"  - Sensor ID: {sensor_id}, Type: {sensor_type}\n"
        field_sensor_details += "\n"

# Compose prompt for generating question-answer pairs
instruction = f"""
Below is a list of fields and their associated sensors in a smart farming system:

{field_sensor_details}

Using the above information, create 20 unique question-answer pairs. Focus on:
1. Retrieving the sensor ID based on sensor type and field.
2. Determining the sensor type or crop type from the sensor ID.
3. Identifying which field a sensor belongs to.
4. Other sensor identification or field-related queries.

Format each entry as a dictionary with 'query' and 'answer' keys:
{{"query": "...", "answer": "..."}}
"""

# Request generation from the LLM
qa_pairs = chat_model.predict(instruction)
print(qa_pairs)



1. {"query": "What is the sensor ID for soil temperature in Field F003?", "answer": "Sensor ID: TEMP-0300"}
2. {"query": "From sensor ID HUM-0500, what is the crop type?", "answer": "Crop: Soybean"}
3. {"query": "Which field does sensor COND-0800 belong to?", "answer": "Field ID: F008, Crop: Potato"}
4. {"query": "What type of sensor is MOIST-0400 used for?", "answer": "Moisture content"}
5. {"query": "In which field is sensor CAM-0700 installed?", "answer": "Field ID: F007, Crop: Coffee"}
6. {"query": "What crop is grown in Field F006?", "answer": "Crop: Soybean"}
7. {"query": "Identify the sensor type for sensor ID CAM-0600.", "answer": "Plant health cameras"}
8. {"query": "Which field utilizes sensor HUM-0300?", "answer": "Field ID: F003, Crop: Sorghum"}
9. {"query": "What is the sensor ID for field air humidity in Field F005?", "answer": "Sensor ID: HUM-0500"}
10. {"query": "From sensor ID MOIST-0900, determine the crop type.", "answer": "Crop: Potato"}
11. {"query": "Which sensor

In [105]:
examples = []

In [ ]:
from langchain.chat_models import ChatOpenAI
import json
import ast


# Initialize the language model
chat_model = ChatOpenAI(api_key=openai_api_key, model="gpt-3.5-turbo")

# Assumed: the detailed 'fields' list (F001 to F006) is already defined

# Construct sensor info string for the LLM prompt
field_sensor_details = ""
for field in fields:
    for field_id, field_info in field.items():
        crop = field_info['crop']
        field_sensor_details += f"Field ID: {field_id}, Crop: {crop}\n"

        for sensor_category, sensors in field_info['sensors'].items():
            for sensor in sensors:
                sensor_id = sensor.get('sensor_id') or sensor.get('camera_id')
                sensor_type = sensor_category.replace('_', ' ').capitalize()
                field_sensor_details += f"  - Sensor ID: {sensor_id}, Type: {sensor_type}\n"
        field_sensor_details += "\n"

# Compose prompt for generating question-answer pairs
instruction = f"""
Below is a list of fields and their associated sensors in a smart farming system:

{field_sensor_details}

Using the above information, create 20 unique question-answer pairs. Focus on:
1. Retrieving the sensor ID based on sensor type and field.
2. Determining the sensor type or crop type from the sensor ID.
3. Identifying which field a sensor belongs to.
4. Other sensor identification or field-related queries.

Format each entry as a dictionary with 'query' and 'answer' keys:
{{"query": "...", "answer": "..."}}
"""

# Request generation from the LLM
qa_pairs_raw = chat_model.predict(instruction)

# Try to safely convert the raw output to a Python list
try:
    qa_pairs = json.loads(qa_pairs_raw)
except json.JSONDecodeError:
    try:
        qa_pairs = ast.literal_eval(qa_pairs_raw)
    except (ValueError, SyntaxError):
        qa_pairs = []
        print("Failed to parse LLM output.")

# Append to the examples list
examples = []
examples.extend(qa_pairs)

# Output result
print(examples)


Failed to parse LLM output.
[]


In [108]:
print("Raw LLM Output:\n", qa_pairs_raw)


Raw LLM Output:
 1. {"query": "What is the sensor ID for soil temperature in Field F003?", "answer": "Sensor ID: TEMP-0300"}
2. {"query": "What type of crop is monitored by sensor CAM-0500?", "answer": "Crop: Soybean"}
3. {"query": "Which field does sensor HUM-0400 belong to?", "answer": "Field ID: F004"}
4. {"query": "What is the sensor type for sensor ID COND-0800?", "answer": "Type: Soil conductivity"}
5. {"query": "What crop is grown in Field F008?", "answer": "Crop: Potato"}
6. {"query": "What is the sensor ID for moisture content in Field F007?", "answer": "Sensor ID: MOIST-0700"}
7. {"query": "Which sensor monitors field air humidity in Field F005?", "answer": "Sensor ID: HUM-0500"}
8. {"query": "What type of sensor is CAM-0600?", "answer": "Type: Plant health cameras"}
9. {"query": "What crop is monitored by sensor HUM-0900?", "answer": "Crop: Potato"}
10. {"query": "Which field uses sensor CAM-0300?", "answer": "Field ID: F003"}
11. {"query": "What is the sensor type for senso

In [111]:
qa_pairs_raw

'1. {"query": "What is the sensor ID for soil temperature in Field F003?", "answer": "Sensor ID: TEMP-0300"}\n2. {"query": "What type of crop is monitored by sensor CAM-0500?", "answer": "Crop: Soybean"}\n3. {"query": "Which field does sensor HUM-0400 belong to?", "answer": "Field ID: F004"}\n4. {"query": "What is the sensor type for sensor ID COND-0800?", "answer": "Type: Soil conductivity"}\n5. {"query": "What crop is grown in Field F008?", "answer": "Crop: Potato"}\n6. {"query": "What is the sensor ID for moisture content in Field F007?", "answer": "Sensor ID: MOIST-0700"}\n7. {"query": "Which sensor monitors field air humidity in Field F005?", "answer": "Sensor ID: HUM-0500"}\n8. {"query": "What type of sensor is CAM-0600?", "answer": "Type: Plant health cameras"}\n9. {"query": "What crop is monitored by sensor HUM-0900?", "answer": "Crop: Potato"}\n10. {"query": "Which field uses sensor CAM-0300?", "answer": "Field ID: F003"}\n11. {"query": "What is the sensor type for sensor ID T

In [121]:
import re

# Extract JSON-like dictionaries using regex
json_like_items = re.findall(r'\{.*?\}', qa_pairs_raw, re.DOTALL)

# Convert them into actual dictionaries
qa_pairs = [json.loads(item) for item in json_like_items]

In [122]:
ex = qa_pairs

In [123]:
ex

[{'query': 'What is the sensor ID for soil temperature in Field F003?',
  'answer': 'Sensor ID: TEMP-0300'},
 {'query': 'What type of crop is monitored by sensor CAM-0500?',
  'answer': 'Crop: Soybean'},
 {'query': 'Which field does sensor HUM-0400 belong to?',
  'answer': 'Field ID: F004'},
 {'query': 'What is the sensor type for sensor ID COND-0800?',
  'answer': 'Type: Soil conductivity'},
 {'query': 'What crop is grown in Field F008?', 'answer': 'Crop: Potato'},
 {'query': 'What is the sensor ID for moisture content in Field F007?',
  'answer': 'Sensor ID: MOIST-0700'},
 {'query': 'Which sensor monitors field air humidity in Field F005?',
  'answer': 'Sensor ID: HUM-0500'},
 {'query': 'What type of sensor is CAM-0600?',
  'answer': 'Type: Plant health cameras'},
 {'query': 'What crop is monitored by sensor HUM-0900?',
  'answer': 'Crop: Potato'},
 {'query': 'Which field uses sensor CAM-0300?', 'answer': 'Field ID: F003'},
 {'query': 'What is the sensor type for sensor ID TEMP-0200?

In [114]:
qa_pairs[0]

{'query': 'What is the sensor ID for soil temperature in Field F003?',
 'answer': 'Sensor ID: TEMP-0300'}

In [115]:
qa_pairs[18]

{'query': 'What is the sensor ID for moisture content in Field F008?',
 'answer': 'Sensor ID: MOIST-0800'}

In [117]:
# Extract the QA pairs
qa_pairs_new = [output['query'] for output in ex]

In [120]:
qa_pairs_new

['What is the sensor ID for soil temperature in Field F003?',
 'What type of crop is monitored by sensor CAM-0500?',
 'Which field does sensor HUM-0400 belong to?',
 'What is the sensor type for sensor ID COND-0800?',
 'What crop is grown in Field F008?',
 'What is the sensor ID for moisture content in Field F007?',
 'Which sensor monitors field air humidity in Field F005?',
 'What type of sensor is CAM-0600?',
 'What crop is monitored by sensor HUM-0900?',
 'Which field uses sensor CAM-0300?',
 'What is the sensor type for sensor ID TEMP-0200?',
 'What crop is grown in Field F001?',
 'What is the sensor ID for field air humidity in Field F006?',
 'Which sensor monitors soil conductivity in Field F009?',
 'What type of crop is monitored by sensor TEMP-0400?',
 'Which field does sensor MOIST-0500 belong to?',
 'What is the sensor type for sensor ID CAM-0700?',
 'What crop is grown in Field F002?',
 'What is the sensor ID for moisture content in Field F008?',
 'Which sensor monitors soil

In [125]:
# Generate model predictions
model_predictions = []
for pair in ex:
    question = pair['query']
    result = retrieval_qa.run(question)
    model_predictions.append({'result': result})


In [126]:
qa_pairs

[{'query': 'What is the sensor ID for soil temperature in Field F003?',
  'answer': 'Sensor ID: TEMP-0300'},
 {'query': 'What type of crop is monitored by sensor CAM-0500?',
  'answer': 'Crop: Soybean'},
 {'query': 'Which field does sensor HUM-0400 belong to?',
  'answer': 'Field ID: F004'},
 {'query': 'What is the sensor type for sensor ID COND-0800?',
  'answer': 'Type: Soil conductivity'},
 {'query': 'What crop is grown in Field F008?', 'answer': 'Crop: Potato'},
 {'query': 'What is the sensor ID for moisture content in Field F007?',
  'answer': 'Sensor ID: MOIST-0700'},
 {'query': 'Which sensor monitors field air humidity in Field F005?',
  'answer': 'Sensor ID: HUM-0500'},
 {'query': 'What type of sensor is CAM-0600?',
  'answer': 'Type: Plant health cameras'},
 {'query': 'What crop is monitored by sensor HUM-0900?',
  'answer': 'Crop: Potato'},
 {'query': 'Which field uses sensor CAM-0300?', 'answer': 'Field ID: F003'},
 {'query': 'What is the sensor type for sensor ID TEMP-0200?

In [127]:
from langchain.evaluation.qa import QAEvalChain

# Create the evaluation chain
eval_chain = QAEvalChain.from_llm(llm)

# Evaluate the predictions
graded_outputs = eval_chain.evaluate(qa_pairs, model_predictions)

# Display the evaluation results
for i, graded in enumerate(graded_outputs):
    print(f"Example {i+1}:")
    print(f"Question: {qa_pairs[i]['query']}")
    print(f"Reference Answer: {qa_pairs[i]['answer']}")
    print(f"Model Prediction: {model_predictions[i]['result']}")
    print(f"Evaluation Result: {graded}")
    print("-" * 50)


Example 1:
Question: What is the sensor ID for soil temperature in Field F003?
Reference Answer: Sensor ID: TEMP-0300
Model Prediction: I don't have the specific information about the sensor ID for soil temperature in Field F003.
Evaluation Result: {'results': 'CORRECT'}
--------------------------------------------------
Example 2:
Question: What type of crop is monitored by sensor CAM-0500?
Reference Answer: Crop: Soybean
Model Prediction: I don't have enough information to determine the specific type of crop monitored by sensor CAM-0500.
Evaluation Result: {'results': 'CORRECT'}
--------------------------------------------------
Example 3:
Question: Which field does sensor HUM-0400 belong to?
Reference Answer: Field ID: F004
Model Prediction: I don't have enough information to determine which field sensor HUM-0400 belongs to.
Evaluation Result: {'results': 'CORRECT'}
--------------------------------------------------
Example 4:
Question: What is the sensor type for sensor ID COND-080

In [76]:
doc ="""
      'status': 'close',
      'linked_valves': ['WV-0100']},
     {'pump_id': 'PUMP-0101',
      'type': 'manual',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['FD-0100']}],
    'water_valves': [{'valve_id': 'WV-0100',
      'type': 'automated',
      'operation_type': 'spray',
      'status': 'open'}],
    'fertilizer_dispensers': [{'valve_id': 'FD-0100',
      'type': 'manual',
      'operation_type': 'drip',
      'status': 'close'}]}}},
 {'F002': {'name': 'Northeast Field',
   'crop': 'Maize',
   'area': '62.5 acres',
   'boundary_gps': {'north': {'lat': 35.6789, 'long': -98.1234},
    'east': {'lat': 35.6789, 'long': -98.1234},
    'south': {'lat': 35.6789, 'long': -98.1234},
    'west': {'lat': 35.6789, 'long': -98.1234}},
   'sensor_list': {'soil_temperature': ['TEMP-0200'],
    'field_air_humidity': ['HUM-0200'],
    'soil_conductivity': ['COND-0200'],
    'moisture_content': ['MOIST-0200'],
    'plant_health': ['CAM-0200']},
   'actuator_list': {'pumps': ['PUMP-0200', 'PUMP-0201'],
    'water_valves': ['WV-0200'],
    'fertilizer_dispensers': ['FD-0200']},
   'sensors': {'soil_temperature': [{'sensor_id': 'TEMP-0200',
      'gps': {'lat': 35.6801, 'long': -98.1201},
      'status': 'transmitting',
      'unit': 'celsius'}],
    'field_air_humidity': [{'sensor_id': 'HUM-0200',
      'gps': {'lat': 35.6802, 'long': -98.1202},
      'status': 'transmitting',
      'unit': 'grams/cubic meter'}],
    'soil_conductivity': [{'sensor_id': 'COND-0200',
      'gps': {'lat': 35.6803, 'long': -98.1203},
      'status': 'transmitting',
      'unit': 'millisiemens/meter'}],
    'moisture_content': [{'sensor_id': 'MOIST-0200',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}],
    'plant_health_cameras': [{'camera_id': 'CAM-0200',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}]},
   'actuators': {'pumps': [{'pump_id': 'PUMP-0200',
      'type': 'automated',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['WV-0200']},
     {'pump_id': 'PUMP-0201',
      'type': 'manual',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['FD-0200']}],
    'water_valves': [{'valve_id': 'WV-0200',
      'type': 'automated',
      'operation_type': 'spray',
      'status': 'open'}],
    'fertilizer_dispensers': [{'valve_id': 'FD-0200',
      'type': 'manual',
      'operation_type': 'drip',
      'status': 'close'}]}}},
 {'F003': {'name': 'East Field',
   'crop': 'Sorghum',
   'area': '62.5 acres',
   'boundary_gps': {'north': {'lat': 35.6789, 'long': -98.1234},
    'east': {'lat': 35.6789, 'long': -98.1234},
    'south': {'lat': 35.6789, 'long': -98.1234},
    'west': {'lat': 35.6789, 'long': -98.1234}},
   'sensor_list': {'soil_temperature': ['TEMP-0300'],
    'field_air_humidity': ['HUM-0300'],
    'soil_conductivity': ['COND-0300'],
    'moisture_content': ['MOIST-0300'],
    'plant_health': ['CAM-0300']},
   'actuator_list': {'pumps': ['PUMP-0300', 'PUMP-0301'],
    'water_valves': ['WV-0300'],
    'fertilizer_dispensers': ['FD-0300']},
   'sensors': {'soil_temperature': [{'sensor_id': 'TEMP-0300',
      'gps': {'lat': 35.6801, 'long': -98.1201},
      'status': 'transmitting',
      'unit': 'celsius'}],
    'field_air_humidity': [{'sensor_id': 'HUM-0300',
      'gps': {'lat': 35.6802, 'long': -98.1202},
      'status': 'transmitting',
      'unit': 'grams/cubic meter'}],
    'soil_conductivity': [{'sensor_id': 'COND-0300',
      'gps': {'lat': 35.6803, 'long': -98.1203},
      'status': 'transmitting',
      'unit': 'millisiemens/meter'}],
    'moisture_content': [{'sensor_id': 'MOIST-0300',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}],
    'plant_health_cameras': [{'camera_id': 'CAM-0300',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}]},
   'actuators': {'pumps': [{'pump_id': 'PUMP-0300',
      'type': 'automated',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['WV-0300']},
     {'pump_id': 'PUMP-0301',
      'type': 'manual',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['FD-0300']}],
    'water_valves': [{'valve_id': 'WV-0300',
      'type': 'automated',
      'operation_type': 'spray',
      'status': 'open'}],
    'fertilizer_dispensers': [{'valve_id': 'FD-0300',
      'type': 'manual',
      'operation_type': 'drip',
      'status': 'close'}]}}},
 {'F004': {'name': 'Southeast Field',
   'crop': 'Maize',
   'area': '62.5 acres',
   'boundary_gps': {'north': {'lat': 35.6789, 'long': -98.1234},
    'east': {'lat': 35.6789, 'long': -98.1234},
    'south': {'lat': 35.6789, 'long': -98.1234},
    'west': {'lat': 35.6789, 'long': -98.1234}},
   'sensor_list': {'soil_temperature': ['TEMP-0400'],
    'field_air_humidity': ['HUM-0400'],
    'soil_conductivity': ['COND-0400'],
    'moisture_content': ['MOIST-0400'],
    'plant_health': ['CAM-0400']},
   'actuator_list': {'pumps': ['PUMP-0400', 'PUMP-0401'],
    'water_valves': ['WV-0400'],
    'fertilizer_dispensers': ['FD-0400']},
   'sensors': {'soil_temperature': [{'sensor_id': 'TEMP-0400',
      'gps': {'lat': 35.6801, 'long': -98.1201},
      'status': 'transmitting',
      'unit': 'celsius'}],
    'field_air_humidity': [{'sensor_id': 'HUM-0400',
      'gps': {'lat': 35.6802, 'long': -98.1202},
      'status': 'transmitting',
      'unit': 'grams/cubic meter'}],
    'soil_conductivity': [{'sensor_id': 'COND-0400',
      'gps': {'lat': 35.6803, 'long': -98.1203},
      'status': 'transmitting',
      'unit': 'millisiemens/meter'}],
    'moisture_content': [{'sensor_id': 'MOIST-0400',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}],
    'plant_health_cameras': [{'camera_id': 'CAM-0400',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}]},
   'actuators': {'pumps': [{'pump_id': 'PUMP-0400',
      'type': 'automated',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['WV-0400']},
     {'pump_id': 'PUMP-0401',
      'type': 'manual',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['FD-0400']}],
    'water_valves': [{'valve_id': 'WV-0400',
      'type': 'automated',
      'operation_type': 'spray',
      'status': 'open'}],
    'fertilizer_dispensers': [{'valve_id': 'FD-0400',
      'type': 'manual',
      'operation_type': 'drip',
      'status': 'close'}]}}},
 {'F005': {'name': 'South Field',
   'crop': 'soybean',
   'area': '62.5 acres',
   'boundary_gps': {'north': {'lat': 35.6789, 'long': -98.1234},
    'east': {'lat': 35.6789, 'long': -98.1234},
    'south': {'lat': 35.6789, 'long': -98.1234},
    'west': {'lat': 35.6789, 'long': -98.1234}},
   'sensor_list': {'soil_temperature': ['TEMP-0500'],
    'field_air_humidity': ['HUM-0500'],
    'soil_conductivity': ['COND-0500'],
    'moisture_content': ['MOIST-0500'],
    'plant_health': ['CAM-0500']},
   'actuator_list': {'pumps': ['PUMP-0500', 'PUMP-0501'],
    'water_valves': ['WV-0500'],
    'fertilizer_dispensers': ['FD-0500']},
   'sensors': {'soil_temperature': [{'sensor_id': 'TEMP-0500',
      'gps': {'lat': 35.6801, 'long': -98.1201},
      'status': 'transmitting',
      'unit': 'celsius'}],
    'field_air_humidity': [{'sensor_id': 'HUM-0500',
      'gps': {'lat': 35.6802, 'long': -98.1202},
      'status': 'transmitting',
      'unit': 'grams/cubic meter'}],
    'soil_conductivity': [{'sensor_id': 'COND-0500',
      'gps': {'lat': 35.6803, 'long': -98.1203},
      'status': 'transmitting',
      'unit': 'millisiemens/meter'}],
    'moisture_content': [{'sensor_id': 'MOIST-0500',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}],
    'plant_health_cameras': [{'camera_id': 'CAM-0500',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}]},
   'actuators': {'pumps': [{'pump_id': 'PUMP-0500',
      'type': 'automated',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['WV-0500']},
     {'pump_id': 'PUMP-0501',
      'type': 'manual',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['FD-0500']}],
    'water_valves': [{'valve_id': 'WV-0500',
      'type': 'automated',
      'operation_type': 'spray',
      'status': 'open'}],
    'fertilizer_dispensers': [{'valve_id': 'FD-0500',
      'type': 'manual',
      'operation_type': 'drip',
      'status': 'close'}]}}},
 {'F006': {'name': 'Southwest Field',
   'crop': 'Soybean',
   'area': '62.5 acres',
   'boundary_gps': {'north': {'lat': 35.6789, 'long': -98.1234},
    'east': {'lat': 35.6789, 'long': -98.1234},
    'south': {'lat': 35.6789, 'long': -98.1234},
    'west': {'lat': 35.6789, 'long': -98.1234}},
   'sensor_list': {'soil_temperature': ['TEMP-0600'],
    'field_air_humidity': ['HUM-0600'],
    'soil_conductivity': ['COND-0600'],
    'moisture_content': ['MOIST-0600'],
    'plant_health': ['CAM-0600']},
   'actuator_list': {'pumps': ['PUMP-0600', 'PUMP-0601'],
    'water_valves': ['WV-0600'],
    'fertilizer_dispensers': ['FD-0600']},
   'sensors': {'soil_temperature': [{'sensor_id': 'TEMP-0600',
      'gps': {'lat': 35.6801, 'long': -98.1201},
      'status': 'transmitting',
      'unit': 'celsius'}],
    'field_air_humidity': [{'sensor_id': 'HUM-0600',
      'gps': {'lat': 35.6802, 'long': -98.1202},
      'status': 'transmitting',
      'unit': 'grams/cubic meter'}],
    'soil_conductivity': [{'sensor_id': 'COND-0600',
      'gps': {'lat': 35.6803, 'long': -98.1203},
      'status': 'transmitting',
      'unit': 'millisiemens/meter'}],
    'moisture_content': [{'sensor_id': 'MOIST-0600',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}],
    'plant_health_cameras': [{'camera_id': 'CAM-0600',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}]},
   'actuators': {'pumps': [{'pump_id': 'PUMP-0600',
      'type': 'automated',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['WV-0600']},
     {'pump_id': 'PUMP-0601',
      'type': 'manual',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['FD-0600']}],
    'water_valves': [{'valve_id': 'WV-0600',
      'type': 'automated',
      'operation_type': 'spray',
      'status': 'open'}],
    'fertilizer_dispensers': [{'valve_id': 'FD-0600',
      'type': 'manual',
      'operation_type': 'drip',
      'status': 'close'}]}}},
 {'F007': {'name': 'West Field',
   'crop': 'Coffee',
   'area': '62.5 acres',
   'boundary_gps': {'north': {'lat': 35.6789, 'long': -98.1234},
    'east': {'lat': 35.6789, 'long': -98.1234},
    'south': {'lat': 35.6789, 'long': -98.1234},
    'west': {'lat': 35.6789, 'long': -98.1234}},
   'sensor_list': {'soil_temperature': ['TEMP-0700'],
    'field_air_humidity': ['HUM-0700'],
    'soil_conductivity': ['COND-0700'],
    'moisture_content': ['MOIST-0700'],
    'plant_health': ['CAM-0700']},
   'actuator_list': {'pumps': ['PUMP-0700', 'PUMP-0701'],
    'water_valves': ['WV-0700'],
    'fertilizer_dispensers': ['FD-0700']},
   'sensors': {'soil_temperature': [{'sensor_id': 'TEMP-0700',
      'gps': {'lat': 35.6801, 'long': -98.1201},
      'status': 'transmitting',
      'unit': 'celsius'}],
    'field_air_humidity': [{'sensor_id': 'HUM-0700',
      'gps': {'lat': 35.6802, 'long': -98.1202},
      'status': 'transmitting',
      'unit': 'grams/cubic meter'}],
    'soil_conductivity': [{'sensor_id': 'COND-0700',
      'gps': {'lat': 35.6803, 'long': -98.1203},
      'status': 'transmitting',
      'unit': 'millisiemens/meter'}],
    'moisture_content': [{'sensor_id': 'MOIST-0700',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}],
    'plant_health_cameras': [{'camera_id': 'CAM-0700',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}]},
   'actuators': {'pumps': [{'pump_id': 'PUMP-0700',
      'type': 'automated',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['WV-0700']},
     {'pump_id': 'PUMP-0701',
      'type': 'manual',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['FD-0700']}],
    'water_valves': [{'valve_id': 'WV-0700',
      'type': 'automated',
      'operation_type': 'spray',
      'status': 'open'}],
    'fertilizer_dispensers': [{'valve_id': 'FD-0700',
      'type': 'manual',
      'operation_type': 'drip',
      'status': 'close'}]}}},
 {'F008': {'name': 'Northwest Field',
   'crop': 'Potato',
   'area': '62.5 acres',
   'boundary_gps': {'north': {'lat': 35.6789, 'long': -98.1234},
    'east': {'lat': 35.6789, 'long': -98.1234},
    'south': {'lat': 35.6789, 'long': -98.1234},
    'west': {'lat': 35.6789, 'long': -98.1234}},
   'sensor_list': {'soil_temperature': ['TEMP-0800'],
    'field_air_humidity': ['HUM-0800'],
    'soil_conductivity': ['COND-0800'],
    'moisture_content': ['MOIST-0800'],
    'plant_health': ['CAM-0800']},
   'actuator_list': {'pumps': ['PUMP-0800', 'PUMP-0801'],
    'water_valves': ['WV-0800'],
    'fertilizer_dispensers': ['FD-0800']},
   'sensors': {'soil_temperature': [{'sensor_id': 'TEMP-0800',
      'gps': {'lat': 35.6801, 'long': -98.1201},
      'status': 'transmitting',
      'unit': 'celsius'}],
    'field_air_humidity': [{'sensor_id': 'HUM-0800',
      'gps': {'lat': 35.6802, 'long': -98.1202},
      'status': 'transmitting',
      'unit': 'grams/cubic meter'}],
    'soil_conductivity': [{'sensor_id': 'COND-0800',
      'gps': {'lat': 35.6803, 'long': -98.1203},
      'status': 'transmitting',
      'unit': 'millisiemens/meter'}],
    'moisture_content': [{'sensor_id': 'MOIST-0800',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}],
    'plant_health_cameras': [{'camera_id': 'CAM-0800',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}]},
   'actuators': {'pumps': [{'pump_id': 'PUMP-0800',
      'type': 'automated',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['WV-0800']},
     {'pump_id': 'PUMP-0801',
      'type': 'manual',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['FD-0800']}],
    'water_valves': [{'valve_id': 'WV-0800',
      'type': 'automated',
      'operation_type': 'spray',
      'status': 'open'}],
    'fertilizer_dispensers': [{'valve_id': 'FD-0800',
      'type': 'manual',
      'operation_type': 'drip',
      'status': 'close'}]}}},
 {'F009': {'name': 'Central Field',
   'crop': 'Potato',
   'area': '62.5 acres',
   'boundary_gps': {'north': {'lat': 35.6789, 'long': -98.1234},
    'east': {'lat': 35.6789, 'long': -98.1234},
    'south': {'lat': 35.6789, 'long': -98.1234},
    'west': {'lat': 35.6789, 'long': -98.1234}},
   'sensor_list': {'soil_temperature': ['TEMP-0900'],
    'field_air_humidity': ['HUM-0900'],
    'soil_conductivity': ['COND-0900'],
    'moisture_content': ['MOIST-0900'],
    'plant_health': ['CAM-0900']},
   'actuator_list': {'pumps': ['PUMP-0900', 'PUMP-0901'],
    'water_valves': ['WV-0900'],
    'fertilizer_dispensers': ['FD-0900']},
   'sensors': {'soil_temperature': [{'sensor_id': 'TEMP-0900',
      'gps': {'lat': 35.6801, 'long': -98.1201},
      'status': 'transmitting',
      'unit': 'celsius'}],
    'field_air_humidity': [{'sensor_id': 'HUM-0900',
      'gps': {'lat': 35.6802, 'long': -98.1202},
      'status': 'transmitting',
      'unit': 'grams/cubic meter'}],
    'soil_conductivity': [{'sensor_id': 'COND-0900',
      'gps': {'lat': 35.6803, 'long': -98.1203},
      'status': 'transmitting',
      'unit': 'millisiemens/meter'}],
    'moisture_content': [{'sensor_id': 'MOIST-0900',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}],
    'plant_health_cameras': [{'camera_id': 'CAM-0900',
      'gps': {'lat': 35.6804, 'long': -98.1204},
      'status': 'transmitting',
      'unit': 'percentage'}]},
   'actuators': {'pumps': [{'pump_id': 'PUMP-0900',
      'type': 'automated',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['WV-0900']},
     {'pump_id': 'PUMP-0901',
      'type': 'manual',
      'base_speed': '500 L/hr',
      'status': 'close',
      'linked_valves': ['FD-0900']}],
    'water_valves': [{'valve_id': 'WV-0900',
      'type': 'automated',
      'operation_type': 'spray',
      'status': 'open'}],
    'fertilizer_dispensers': [{'valve_id': 'FD-0900',
      'type': 'manual',
      'operation_type': 'drip',
      'status': 'close'}]}}}]

=== Field ID: F001 ===
Name: North Field
Crop: Maize
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0100, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0100, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0100, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0100, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0100, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0100, Type: automated, Status: close
      Linked Valves: WV-0100
      Base Speed: 500 L/hr
    ID: PUMP-0101, Type: manual, Status: close
      Linked Valves: FD-0100
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0100, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0100, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F002 ===
Name: Northeast Field
Crop: Maize
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0200, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0200, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0200, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0200, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0200, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0200, Type: automated, Status: close
      Linked Valves: WV-0200
      Base Speed: 500 L/hr
    ID: PUMP-0201, Type: manual, Status: close
      Linked Valves: FD-0200
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0200, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0200, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F003 ===
Name: East Field
Crop: Sorghum
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0300, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0300, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0300, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0300, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0300, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0300, Type: automated, Status: close
      Linked Valves: WV-0300
      Base Speed: 500 L/hr
    ID: PUMP-0301, Type: manual, Status: close
      Linked Valves: FD-0300
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0300, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0300, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F004 ===
Name: Southeast Field
Crop: Maize
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0400, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0400, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0400, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0400, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0400, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0400, Type: automated, Status: close
      Linked Valves: WV-0400
      Base Speed: 500 L/hr
    ID: PUMP-0401, Type: manual, Status: close
      Linked Valves: FD-0400
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0400, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0400, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F005 ===
Name: South Field
Crop: soybean
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0500, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0500, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0500, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0500, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0500, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0500, Type: automated, Status: close
      Linked Valves: WV-0500
      Base Speed: 500 L/hr
    ID: PUMP-0501, Type: manual, Status: close
      Linked Valves: FD-0500
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0500, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0500, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F006 ===
Name: Southwest Field
Crop: Soybean
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0600, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0600, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0600, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0600, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0600, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0600, Type: automated, Status: close
      Linked Valves: WV-0600
      Base Speed: 500 L/hr
    ID: PUMP-0601, Type: manual, Status: close
      Linked Valves: FD-0600
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0600, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0600, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F007 ===
Name: West Field
Crop: Coffee
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0700, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0700, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0700, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0700, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0700, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0700, Type: automated, Status: close
      Linked Valves: WV-0700
      Base Speed: 500 L/hr
    ID: PUMP-0701, Type: manual, Status: close
      Linked Valves: FD-0700
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0700, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0700, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F008 ===
Name: Northwest Field
Crop: Potato
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0800, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0800, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0800, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0800, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0800, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0800, Type: automated, Status: close
      Linked Valves: WV-0800
      Base Speed: 500 L/hr
    ID: PUMP-0801, Type: manual, Status: close
      Linked Valves: FD-0800
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0800, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0800, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F009 ===
Name: Central Field
Crop: Potato
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0900, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0900, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0900, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0900, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0900, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0900, Type: automated, Status: close
      Linked Valves: WV-0900
      Base Speed: 500 L/hr
    ID: PUMP-0901, Type: manual, Status: close
      Linked Valves: FD-0900
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0900, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0900, Type: manual, Status: close
      Operation Type: drip"""

In [177]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.document_loaders import UnstructuredFileLoader
from langchain_community.vectorstores import FAISS
from langchain_openai.chat_models import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [178]:
# loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
# data = loader.load()
file_path = "document.txt"  # Specify the path where you'd like to save the file
# with open(file_path, "w") as f:
#     f.write(file_path)  # Save the content of 'doc' to a text file

# Use TextLoader to load text-based documents
# loader = UnstructuredFileLoader(file_path)
loader = TextLoader(file_path,  encoding="utf-8")

document = loader.load()


In [179]:
# pip install faiss-cpu 

In [180]:
# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
all_splits = text_splitter.split_documents(document)

# # Store splits
vectorstore = FAISS.from_documents(documents=all_splits, embedding=OpenAIEmbeddings(api_key=openai_api_key))

In [181]:
retriever = vectorstore.as_retriever()

qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(api_key=openai_api_key),
    retriever=retriever,
    return_source_documents=True  # Optional, helpful for transparency
)


In [182]:
query = "What the temperature sensor for south field?"
# response = qa_chain.run(query)
response = qa_chain(query)


print(response)


{'query': 'What the temperature sensor for south field?', 'result': 'The temperature sensor for the South Field (Field ID: F003) is TEMP-0100.', 'source_documents': [Document(id='ee6cad52-9d34-494b-b22b-9ba9eca6849a', metadata={'source': 'document.txt'}, page_content='Sensors:\nSoil_temperature:\nID: TEMP-0200, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)\nField_air_humidity:\nID: HUM-0200, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)\nSoil_conductivity:\nID: COND-0200, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)\nMoisture_content:\nID: MOIST-0200, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)\nPlant_health_cameras:'), Document(id='b2c8e2ad-5635-452b-b58d-17a22164c03b', metadata={'source': 'document.txt'}, page_content='Sensors:\nSoil_temperature:\nID: TEMP-0300, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)\nField_air_humidity:\nID: HUM-0300, Status: transmitting, Unit: grams/cu

In [186]:
query = "What information about the field do you have?"
# response = qa_chain.run(query)
response = qa_chain(query)


print(response)


{'query': 'What information about the field do you have?', 'result': 'I have information about three fields:\n1. North Field - Maize - 62.5 acres\n2. Northeast Field - Maize - 62.5 acres\n3. East Field - Sorghum - 62.5 acres\n\nI also have the GPS coordinates for their boundaries and information about sensors in the area.', 'source_documents': [Document(id='00a7b10c-d45e-47ef-8b5e-6b1b21b7f2aa', metadata={'source': 'document.txt'}, page_content='=== Field ID: F001 ===\nName: North Field\nCrop: Maize\nArea: 62.5 acres\nBoundary GPS:\nNorth: Lat 35.6789, Long -98.1234\nEast: Lat 35.6789, Long -98.1234\nSouth: Lat 35.6789, Long -98.1234\nWest: Lat 35.6789, Long -98.1234'), Document(id='7a25289c-f8f7-4a3e-ba4f-9bda6d23eef5', metadata={'source': 'document.txt'}, page_content='=== Field ID: F002 ===\nName: Northeast Field\nCrop: Maize\nArea: 62.5 acres\nBoundary GPS:\nNorth: Lat 35.6789, Long -98.1234\nEast: Lat 35.6789, Long -98.1234\nSouth: Lat 35.6789, Long -98.1234\nWest: Lat 35.6789, Lo

In [187]:
for doc in documents:
    print("----")
    print("Metadata:", doc.metadata)
    print("Page Content (first 200 chars):", doc.page_content[:200])


----
Metadata: {'source': 'document.txt'}
Page Content (first 200 chars): === Field ID: F001 ===
Name: North Field
Crop: Maize
Area: 62.5 acres
Boundary GPS:
North: Lat 35.6789, Long -98.1234
East: Lat 35.6789, Long -98.1234
South: Lat 35.6789, Long -98.1234
West: Lat 35.67


In [185]:
for doc in document:
    print(f"Content: {doc.page_content}")  # Access the textual content
    print(f"Metadata: {doc.metadata}")      # Access any metadata associated with the document

Content: === Field ID: F001 ===
Name: North Field
Crop: Maize
Area: 62.5 acres
Boundary GPS:
North: Lat 35.6789, Long -98.1234
East: Lat 35.6789, Long -98.1234
South: Lat 35.6789, Long -98.1234
West: Lat 35.6789, Long -98.1234

Sensors:
Soil_temperature:
ID: TEMP-0100, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
Field_air_humidity:
ID: HUM-0100, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
Soil_conductivity:
ID: COND-0100, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
Moisture_content:
ID: MOIST-0100, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
Plant_health_cameras:
ID: CAM-0100, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
Pumps:
ID: PUMP-0100, Type: automated, Status: close
Linked Valves: WV-0100
Base Speed: 500 L/hr
ID: PUMP-0101, Type: manual, Status: close
Linked Valves: FD-0100
Base Speed: 500 L/hr
Water_valves:
ID: WV-0100, Type: automated, Status: open
Op

In [112]:
# !pip install unstructured

In [106]:
data

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final resu

In [ ]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import TextLoader  # Correct loader for text
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch
from langchain.embeddings import OpenAIEmbeddings  # Add embedding model
import os

# Save the document content to a temporary text file
file_path = "document.txt"  # Specify the path where you'd like to save the file
# with open(file_path, "w") as f:
#     f.write(file_path)  # Save the content of 'doc' to a text file

# Use TextLoader to load text-based documents
loader = TextLoader(file_path)

# Initialize the embedding model
embedding = OpenAIEmbeddings(api_key=openai_api_key)

# Create an index using DocArrayInMemorySearch with the embedding model
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embedding  # Add embedding here
).from_loaders([loader])

# Initialize the LLM model
llm = ChatOpenAI(api_key=openai_api_key, model="gpt-3.5-turbo", temperature=0.0)

# Create a RetrievalQA chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=index.vectorstore.as_retriever(),
    verbose=True,
    chain_type_kwargs={
        "document_separator": "<<<<>>>>>"
    }
)

In [8]:
data_doc = loader.load()
data_doc

[Document(metadata={'source': 'document.txt'}, page_content='document.txt')]

In [37]:
from langchain.evaluation.qa import QAGenerateChain
llm_model = "gpt-3.5-turbo"
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(api_key=openai_api_key,model=llm_model))
# example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(model=llm_model))

In [ ]:
# new_examples = example_gen_chain.apply_and_parse(
#     [{"doc": t} for t in data[:5]]
# )

/Users/george/Downloads/eai_methods/env/lib/python3.10/site-packages/langchain/chains/llm.py:369: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


In [69]:
for doc in data_doc:
    print(doc.page_content)
    print(doc.metadata)


document.txt
{'source': 'document.txt'}


In [83]:
# new_examples = example_gen_chain.apply_and_parse(data_doc)
# Assuming 'data_doc' is a list of Document objects
new_examples = example_gen_chain.apply_and_parse([{"doc": doc.page_content} for doc in data_doc])


/Users/george/Downloads/eai_methods/env/lib/python3.10/site-packages/langchain/chains/llm.py:369: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


In [85]:
new_examples

[{'qa_pairs': {'query': 'According to the document, what are the three main types of clouds?',
   'answer': 'The three main types of clouds mentioned in the document are cirrus clouds, cumulus clouds, and stratus clouds.'}}]

In [55]:
new_examples[0]['qa_pairs']['query']

'What is the crop and area of the Northeast Field (F002) based on the provided document?'

In [86]:
import langchain
# Turn off the debug mode
langchain.debug = False

In [88]:
formatted_data = [{"doc": doc.page_content} for doc in data_doc]
formatted_data

[{'doc': 'document.txt'}]

In [89]:
new_examples = example_gen_chain.apply_and_parse(formatted_data)

/Users/george/Downloads/eai_methods/env/lib/python3.10/site-packages/langchain/chains/llm.py:369: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


In [90]:
new_examples

[{'qa_pairs': {'query': 'According to the document, what are the three main stages of photosynthesis?',
   'answer': 'The three main stages of photosynthesis, as mentioned in the document, are light-dependent reactions, the Calvin cycle, and photorespiration.'}}]

In [92]:
import json 
with open("farm_model_small.json", "r") as file:
    farm_data = json.load(file)

In [93]:
import langchain 
langchain.debug = True
docs = [{"doc": json.dumps(farm_data['farm']['fields'][i])} for i in range(len(farm_data['farm']['fields']))]
docs

[{'doc': '{"F001": {"name": "North Field", "crop": "Maize", "area": "62.5 acres", "boundary_gps": {"north": {"lat": 35.6789, "long": -98.1234}, "east": {"lat": 35.6789, "long": -98.1234}, "south": {"lat": 35.6789, "long": -98.1234}, "west": {"lat": 35.6789, "long": -98.1234}}, "sensor_list": {"soil_temperature": ["TEMP-0100"], "field_air_humidity": ["HUM-0100"], "soil_conductivity": ["COND-0100"], "moisture_content": ["MOIST-0100"], "plant_health": ["CAM-0100"]}, "actuator_list": {"pumps": ["PUMP-0100", "PUMP-0101"], "water_valves": ["WV-0100"], "fertilizer_dispensers": ["FD-0100"]}, "sensors": {"soil_temperature": [{"sensor_id": "TEMP-0100", "gps": {"lat": 35.6801, "long": -98.1201}, "status": "transmitting", "unit": "celsius"}], "field_air_humidity": [{"sensor_id": "HUM-0100", "gps": {"lat": 35.6802, "long": -98.1202}, "status": "transmitting", "unit": "grams/cubic meter"}], "soil_conductivity": [{"sensor_id": "COND-0100", "gps": {"lat": 35.6803, "long": -98.1203}, "status": "transmi

In [94]:
new_examples = example_gen_chain.apply_and_parse(docs)

/Users/george/Downloads/eai_methods/env/lib/python3.10/site-packages/langchain/chains/llm.py:369: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


[chain/start] [chain:QAGenerateChain] Entering Chain run with input:
{
  "input_list": [
    {
      "doc": "{\"F001\": {\"name\": \"North Field\", \"crop\": \"Maize\", \"area\": \"62.5 acres\", \"boundary_gps\": {\"north\": {\"lat\": 35.6789, \"long\": -98.1234}, \"east\": {\"lat\": 35.6789, \"long\": -98.1234}, \"south\": {\"lat\": 35.6789, \"long\": -98.1234}, \"west\": {\"lat\": 35.6789, \"long\": -98.1234}}, \"sensor_list\": {\"soil_temperature\": [\"TEMP-0100\"], \"field_air_humidity\": [\"HUM-0100\"], \"soil_conductivity\": [\"COND-0100\"], \"moisture_content\": [\"MOIST-0100\"], \"plant_health\": [\"CAM-0100\"]}, \"actuator_list\": {\"pumps\": [\"PUMP-0100\", \"PUMP-0101\"], \"water_valves\": [\"WV-0100\"], \"fertilizer_dispensers\": [\"FD-0100\"]}, \"sensors\": {\"soil_temperature\": [{\"sensor_id\": \"TEMP-0100\", \"gps\": {\"lat\": 35.6801, \"long\": -98.1201}, \"status\": \"transmitting\", \"unit\": \"celsius\"}], \"field_air_humidity\": [{\"sensor_id\": \"HUM-0100\", \"gps

In [95]:
new_examples

[{'qa_pairs': {'query': 'What is the name of the field mentioned in the document and what crop is being grown there?',
   'answer': 'The name of the field is "North Field" and the crop being grown there is Maize.'}},
 {'qa_pairs': {'query': 'What is the crop being grown in the Northeast Field and what are the sensor types and actuator types used in that field?',
   'answer': 'The crop being grown in the Northeast Field is Maize. The sensor types used in the field are soil temperature, field air humidity, soil conductivity, moisture content, and plant health cameras. The actuator types used in the field are pumps, water valves, and fertilizer dispensers.'}},
 {'qa_pairs': {'query': 'What is the name of the field mentioned in the document?',
   'answer': 'East Field'}},
 {'qa_pairs': {'query': 'What is the name of the field mentioned in the document and what crop is being grown there?',
   'answer': 'The field is called "Southeast Field" and maize is being grown there.'}},
 {'qa_pairs': 

In [96]:
# Turn off the debug mode
langchain.debug = False

In [98]:
from langchain.evaluation.qa import QAEvalChain

# Create the evaluation chain
eval_chain = QAEvalChain.from_llm(llm)

In [99]:
# Extract the QA pairs
qa_pairs = [output['qa_pairs'] for output in new_examples]

In [100]:
qa_pairs

[{'query': 'What is the name of the field mentioned in the document and what crop is being grown there?',
  'answer': 'The name of the field is "North Field" and the crop being grown there is Maize.'},
 {'query': 'What is the crop being grown in the Northeast Field and what are the sensor types and actuator types used in that field?',
  'answer': 'The crop being grown in the Northeast Field is Maize. The sensor types used in the field are soil temperature, field air humidity, soil conductivity, moisture content, and plant health cameras. The actuator types used in the field are pumps, water valves, and fertilizer dispensers.'},
 {'query': 'What is the name of the field mentioned in the document?',
  'answer': 'East Field'},
 {'query': 'What is the name of the field mentioned in the document and what crop is being grown there?',
  'answer': 'The field is called "Southeast Field" and maize is being grown there.'},
 {'query': 'What is the crop being grown in South Field and how many acres

In [188]:
# Generate model predictions
model_predictions = []
for pair in qa_pairs:
    # print(pair)
    question = pair['query']
    result = qa.run(question)
    model_predictions.append({'result': result})




> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


In [103]:
model_predictions

[{'result': "I'm sorry, but I do not have access to the content of the document.txt."},
 {'result': "I'm sorry, but I do not have access to the specific details mentioned in the document.txt."},
 {'result': "I'm sorry, but I do not have access to the content of the document. If you provide me with more specific information or details, I may be able to help you further."},
 {'result': "I'm sorry, but I do not have access to the content of the document.txt."},
 {'result': "I'm sorry, but I don't have that specific information in the document provided."},
 {'result': "I'm sorry, but I do not have access to the content of the document.txt."},
 {'result': "I'm sorry, but I don't have that specific information in the document provided."},
 {'result': "I don't have that information."},
 {'result': "I'm sorry, but I don't have that information in the document provided."}]

In [ ]:
from langchain.evaluation.qa import QAEvalChain

# Create the evaluation chain
eval_chain = QAEvalChain.from_llm(llm)

# Evaluate the predictions
graded_outputs = eval_chain.evaluate(qa_pairs, model_predictions)

# Display the evaluation results
for i, graded in enumerate(graded_outputs):
    print(f"Example {i+1}:")
    print(f"Question: {qa_pairs[i]['query']}")
    print(f"Reference Answer: {qa_pairs[i]['answer']}")
    print(f"Model Prediction: {model_predictions[i]['result']}")
    print(f"Evaluation Result: {graded}")
    print("-" * 50)


In [77]:
doc = """=== Field ID: F001 ===
Name: North Field
Crop: Maize
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0100, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0100, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0100, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0100, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0100, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0100, Type: automated, Status: close
      Linked Valves: WV-0100
      Base Speed: 500 L/hr
    ID: PUMP-0101, Type: manual, Status: close
      Linked Valves: FD-0100
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0100, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0100, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F002 ===
Name: Northeast Field
Crop: Maize
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0200, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0200, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0200, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0200, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0200, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0200, Type: automated, Status: close
      Linked Valves: WV-0200
      Base Speed: 500 L/hr
    ID: PUMP-0201, Type: manual, Status: close
      Linked Valves: FD-0200
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0200, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0200, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F003 ===
Name: East Field
Crop: Sorghum
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0300, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0300, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0300, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0300, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0300, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0300, Type: automated, Status: close
      Linked Valves: WV-0300
      Base Speed: 500 L/hr
    ID: PUMP-0301, Type: manual, Status: close
      Linked Valves: FD-0300
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0300, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0300, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F004 ===
Name: Southeast Field
Crop: Maize
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0400, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0400, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0400, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0400, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0400, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0400, Type: automated, Status: close
      Linked Valves: WV-0400
      Base Speed: 500 L/hr
    ID: PUMP-0401, Type: manual, Status: close
      Linked Valves: FD-0400
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0400, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0400, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F005 ===
Name: South Field
Crop: soybean
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0500, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0500, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0500, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0500, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0500, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0500, Type: automated, Status: close
      Linked Valves: WV-0500
      Base Speed: 500 L/hr
    ID: PUMP-0501, Type: manual, Status: close
      Linked Valves: FD-0500
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0500, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0500, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F006 ===
Name: Southwest Field
Crop: Soybean
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0600, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0600, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0600, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0600, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0600, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0600, Type: automated, Status: close
      Linked Valves: WV-0600
      Base Speed: 500 L/hr
    ID: PUMP-0601, Type: manual, Status: close
      Linked Valves: FD-0600
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0600, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0600, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F007 ===
Name: West Field
Crop: Coffee
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0700, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0700, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0700, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0700, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0700, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0700, Type: automated, Status: close
      Linked Valves: WV-0700
      Base Speed: 500 L/hr
    ID: PUMP-0701, Type: manual, Status: close
      Linked Valves: FD-0700
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0700, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0700, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F008 ===
Name: Northwest Field
Crop: Potato
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0800, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0800, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0800, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0800, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0800, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0800, Type: automated, Status: close
      Linked Valves: WV-0800
      Base Speed: 500 L/hr
    ID: PUMP-0801, Type: manual, Status: close
      Linked Valves: FD-0800
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0800, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0800, Type: manual, Status: close
      Operation Type: drip

=== Field ID: F009 ===
Name: Central Field
Crop: Potato
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0900, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0900, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0900, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0900, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0900, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0900, Type: automated, Status: close
      Linked Valves: WV-0900
      Base Speed: 500 L/hr
    ID: PUMP-0901, Type: manual, Status: close
      Linked Valves: FD-0900
      Base Speed: 500 L/hr
  Water_valves:
    ID: WV-0900, Type: automated, Status: open
      Operation Type: spray
  Fertilizer_dispensers:
    ID: FD-0900, Type: manual, Status: close
      Operation Type: drip"""

In [78]:
from langchain.document_loaders import TextLoader

# Create a dummy text file for demonstration
with open("my_document.txt", "w") as f:
    f.write(doc)
    # f.write("This is the content of my first document.\n")
    # f.write("It has two lines of text.")

# Specify the path to your text file
file_path = "my_document.txt"

# Create a TextLoader instance
loader = TextLoader(file_path)

# Load the document(s)
documents = loader.load()

# Access and print the content and metadata
if documents:
    document = documents[0]
    print("Content:")
    print(document.page_content)
    print("\nMetadata:")
    print(document.metadata)
else:
    print("No documents loaded.")

# # Clean up the dummy file (optional)
# import os
# os.remove("my_document.txt")

Content:
=== Field ID: F001 ===
Name: North Field
Crop: Maize
Area: 62.5 acres
Boundary GPS:
  North: Lat 35.6789, Long -98.1234
  East: Lat 35.6789, Long -98.1234
  South: Lat 35.6789, Long -98.1234
  West: Lat 35.6789, Long -98.1234

Sensors:
  Soil_temperature:
    ID: TEMP-0100, Status: transmitting, Unit: celsius, GPS: (35.6801, -98.1201)
  Field_air_humidity:
    ID: HUM-0100, Status: transmitting, Unit: grams/cubic meter, GPS: (35.6802, -98.1202)
  Soil_conductivity:
    ID: COND-0100, Status: transmitting, Unit: millisiemens/meter, GPS: (35.6803, -98.1203)
  Moisture_content:
    ID: MOIST-0100, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)
  Plant_health_cameras:
    ID: CAM-0100, Status: transmitting, Unit: percentage, GPS: (35.6804, -98.1204)

Actuators:
  Pumps:
    ID: PUMP-0100, Type: automated, Status: close
      Linked Valves: WV-0100
      Base Speed: 500 L/hr
    ID: PUMP-0101, Type: manual, Status: close
      Linked Valves: FD-0100
      Base Spe

In [ ]:
# from langchain_community.document_loaders import TextLoader
# from langchain_openai import OpenAIEmbeddings
# from langchain_text_splitters import CharacterTextSplitter

# # Load the document, split it into chunks, embed each chunk and load it into the vector store.
# raw_documents = TextLoader('document.txt').load()
# text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
# documents = text_splitter.split_documents(raw_documents)

In [ ]:
# documents

[Document(metadata={'source': 'document.txt'}, page_content='document.txt')]

In [193]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("document.txt")

documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
embeddings = OpenAIEmbeddings(api_key=openai_api_key)

vectorstore = FAISS.from_documents(texts, embeddings)

In [167]:
retriever = vectorstore.as_retriever()

In [173]:
# docs = retriever.invoke("what is the temperature sensor in south?")
docs = retriever.invoke("Give all sensors in East field?")

In [191]:
print(docs[2].page_content)

Actuators:
Pumps:
ID: PUMP-0100, Type: automated, Status: close
Linked Valves: WV-0100
Base Speed: 500 L/hr
ID: PUMP-0101, Type: manual, Status: close
Linked Valves: FD-0100
Base Speed: 500 L/hr
Water_valves:
ID: WV-0100, Type: automated, Status: open
Operation Type: spray
Fertilizer_dispensers:
ID: FD-0100, Type: manual, Status: close
Operation Type: drip

=== Field ID: F002 ===
Name: Northeast Field
Crop: Maize
Area: 62.5 acres
Boundary GPS:
North: Lat 35.6789, Long -98.1234
East: Lat 35.6789, Long -98.1234
South: Lat 35.6789, Long -98.1234
West: Lat 35.6789, Long -98.1234


In [160]:
len(docs)

1

In [175]:
# # Helper function for printing docs


# def pretty_print_docs(docs):
#     print(
#         f"\n{'-' * 100}\n".join(
#             [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
#         )
#     )

In [176]:
# from langchain_community.document_loaders import TextLoader
# from langchain_community.vectorstores import FAISS
# from langchain_openai import OpenAIEmbeddings
# from langchain_text_splitters import CharacterTextSplitter

# documents = TextLoader("state_of_the_union.txt").load()
# text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
# texts = text_splitter.split_documents(documents)
# retriever = FAISS.from_documents(texts, OpenAIEmbeddings()).as_retriever()

# docs = retriever.invoke("What did the president say about Ketanji Brown Jackson")
# pretty_print_docs(docs)

In [145]:
# !pip install langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 1.6 MB/s eta 0:00:00m eta 0:00:010:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.6/433.6 kB 5.9 MB/s eta 0:00:00m eta 0:00:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 4.8 MB/s eta 0:00:006.9 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 5.1 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.6/33.6 MB 5.6 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.0/499.0 kB 5.6 MB/s eta 0:00:00 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ...